# 05 — Robot Exit Parameter Analysis

Sinyal parametreleri baseline değerlerinde sabit tutulur:

- Buy score: 11
- Minimum ADX: 20
- Volume multiplier: 1.30

Bu notebook yalnızca stop ve trailing parametrelerini geliştirme döneminde test eder.
Holdout dönemi parametre seçiminde kullanılmaz.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src").is_dir()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

from src.config import StrategyConfig, PortfolioConfig
from src.features import add_indicators
from src.signals import build_market_regime
from src.experiments import (
    run_strategy_grid,
    apply_robustness_filters,
    compare_periods,
)

sns.set_theme(style="whitegrid")


## 1. Verileri ve özellikleri hazırla


In [ ]:
stock_prices = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "bist100_robot_clean.parquet"
)

market_prices = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "xu100_robot_clean.parquet"
)

stock_features = add_indicators(stock_prices)
market_features = add_indicators(market_prices)
market_regime = build_market_regime(market_features)

PERIODS = {
    "Development": ("2018-01-01", "2022-12-31"),
    "Validation": ("2023-01-01", "2024-12-31"),
    "Holdout": (
        "2025-01-01",
        stock_features["Date"].max().strftime("%Y-%m-%d"),
    ),
}

base_strategy = StrategyConfig(
    buy_score=12,
    minimum_adx=18.0,
    volume_multiplier=1.3,
)

base_portfolio = PortfolioConfig()

print("Hisse özellikleri:", stock_features.shape)
print("Tarih:", stock_features["Date"].min(), "→", stock_features["Date"].max())


## 2. Çıkış parametre grid'i

İlk aşamada 27 kombinasyon test edilir:

- Initial stop: 1.5 / 2.0 / 2.5 ATR
- Trailing stop: 2.0 / 2.5 / 3.0 ATR
- Trailing aktivasyonu: %4 / %6 / %8

Risk, komisyon, slippage ve giriş sinyalleri sabittir.


In [ ]:
exit_grid = {
    "initial_stop_atr": [1.5, 2.0, 2.5],
    "trailing_stop_atr": [2.0, 2.5, 3.0],
    "trailing_activation_return": [0.04, 0.06, 0.08],
}

exit_development_results = run_strategy_grid(
    stock_features=stock_features,
    market_regime=market_regime,
    base_strategy=base_strategy,
    portfolio_config=base_portfolio,
    parameter_grid=exit_grid,
    start=PERIODS["Development"][0],
    end=PERIODS["Development"][1],
)

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

exit_development_results.to_csv(
    RESULTS_DIR / "exit_grid_development.csv",
    index=False,
)

print("Deney sayısı:", len(exit_development_results))


## 3. Geliştirme döneminde dayanıklılık filtreleri


In [ ]:
robust_exit_development = apply_robustness_filters(
    exit_development_results,
    minimum_trades=150,
    maximum_drawdown_limit=-35.0,
    minimum_profit_factor=1.20,
)

exit_columns = [
    "Experiment_ID",
    "Strategy_initial_stop_atr",
    "Strategy_trailing_stop_atr",
    "Strategy_trailing_activation_return",
    "CAGR_%",
    "Max_Drawdown_%",
    "Profit_Factor",
    "Sharpe",
    "Calmar",
    "Trade_Count",
]

display(
    robust_exit_development[exit_columns].head(15)
)


## 4. Baseline dahil en iyi adayları doğrulama dönemine taşı

Baseline kombinasyonu:

- Initial stop: 2 ATR
- Trailing stop: 2.5 ATR
- Aktivasyon: %6


In [ ]:
baseline_mask = (
    exit_development_results["Strategy_initial_stop_atr"].eq(2.0)
    & exit_development_results["Strategy_trailing_stop_atr"].eq(2.5)
    & exit_development_results[
        "Strategy_trailing_activation_return"
    ].eq(0.06)
)

baseline_exit_row = exit_development_results.loc[
    baseline_mask
].copy()

selected_exit_development = (
    pd.concat(
        [
            robust_exit_development.head(8),
            baseline_exit_row,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=[
            "Strategy_initial_stop_atr",
            "Strategy_trailing_stop_atr",
            "Strategy_trailing_activation_return",
        ]
    )
    .reset_index(drop=True)
)

display(selected_exit_development[exit_columns])


In [ ]:
exit_parameter_columns = [
    "Strategy_initial_stop_atr",
    "Strategy_trailing_stop_atr",
    "Strategy_trailing_activation_return",
]

exit_comparison = compare_periods(
    stock_features=stock_features,
    market_regime=market_regime,
    configurations=selected_exit_development,
    base_strategy=base_strategy,
    portfolio_config=base_portfolio,
    periods={
        "Development": PERIODS["Development"],
        "Validation": PERIODS["Validation"],
    },
    parameter_columns=exit_parameter_columns,
)

exit_comparison.to_csv(
    RESULTS_DIR / "exit_grid_selected_validation.csv",
    index=False,
)

display(
    exit_comparison[
        [
            "Selected_Config",
            "Period_Name",
            "Strategy_initial_stop_atr",
            "Strategy_trailing_stop_atr",
            "Strategy_trailing_activation_return",
            "CAGR_%",
            "Max_Drawdown_%",
            "Profit_Factor",
            "Sharpe",
            "Calmar",
            "Trade_Count",
        ]
    ].sort_values(
        ["Selected_Config", "Period_Name"]
    )
)


## 5. Konservatif dayanıklılık puanı

Seçim yalnızca geliştirme CAGR'ına göre yapılmaz.

- `Robust_Calmar`: geliştirme ve doğrulama Calmar değerlerinin küçüğü
- `Robust_Profit_Factor`: iki dönemdeki Profit Factor değerlerinin küçüğü
- `CAGR_Gap`: dönemler arasındaki mutlak CAGR farkı

Öncelik, iki dönemde de iyi kalan konfigürasyondadır.


In [ ]:
period_metrics = (
    exit_comparison.pivot_table(
        index=[
            "Selected_Config",
            "Strategy_initial_stop_atr",
            "Strategy_trailing_stop_atr",
            "Strategy_trailing_activation_return",
        ],
        columns="Period_Name",
        values=[
            "CAGR_%",
            "Max_Drawdown_%",
            "Profit_Factor",
            "Sharpe",
            "Calmar",
            "Trade_Count",
        ],
        aggfunc="first",
    )
)

period_metrics.columns = [
    f"{metric}_{period}"
    for metric, period in period_metrics.columns
]

exit_stability = period_metrics.reset_index()

exit_stability["Robust_Calmar"] = exit_stability[
    ["Calmar_Development", "Calmar_Validation"]
].min(axis=1)

exit_stability["Robust_Profit_Factor"] = exit_stability[
    ["Profit_Factor_Development", "Profit_Factor_Validation"]
].min(axis=1)

exit_stability["Robust_Sharpe"] = exit_stability[
    ["Sharpe_Development", "Sharpe_Validation"]
].min(axis=1)

exit_stability["CAGR_Gap"] = (
    exit_stability["CAGR_%_Development"]
    - exit_stability["CAGR_%_Validation"]
).abs()

exit_stability["Worst_Drawdown"] = exit_stability[
    ["Max_Drawdown_%_Development", "Max_Drawdown_%_Validation"]
].min(axis=1)

exit_stability = exit_stability.loc[
    exit_stability["Profit_Factor_Validation"].ge(1.30)
    & exit_stability["CAGR_%_Validation"].gt(0)
    & exit_stability["Worst_Drawdown"].ge(-35.0)
].sort_values(
    [
        "Robust_Calmar",
        "Robust_Profit_Factor",
        "Robust_Sharpe",
        "CAGR_Gap",
    ],
    ascending=[False, False, False, True],
).reset_index(drop=True)

display(exit_stability)


## 6. Geliştirme ve doğrulama Calmar karşılaştırması


In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=exit_stability,
    x="Calmar_Development",
    y="Calmar_Validation",
    size="Robust_Profit_Factor",
    hue="Strategy_initial_stop_atr",
    sizes=(80, 300),
)

max_axis = max(
    exit_stability["Calmar_Development"].max(),
    exit_stability["Calmar_Validation"].max(),
)

plt.plot(
    [0, max_axis],
    [0, max_axis],
    linestyle="--",
)

plt.title("Exit Konfigürasyonları — Development vs Validation Calmar")
plt.xlabel("Development Calmar")
plt.ylabel("Validation Calmar")
plt.tight_layout()
plt.show()


## 7. Seçim

İlk satır otomatik olarak kesin kazanan değildir. Baseline ile karşılaştır:

- Validation Calmar
- Validation Profit Factor
- Validation drawdown
- Development-validation farkı
- İşlem sayısı

Seçim yapıldıktan sonra yalnızca o konfigürasyon holdout döneminde bir kez çalıştırılır.
